# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** CTR / Engagement Opportunity Scoring (set provisionally in ML-02, `w01_research_question.ipynb`).

**Task type: ranking / scoring.** Not classification, not clustering.

The reason is the shape of the decision, not the shape of the data. An editor has roughly 50 review hours against ~12,000 candidate pages. Nobody ever asks "is page X under-performing, yes or no?" — they ask "what do I open first?" The output that matches that question is an **ordering**, and the only part of the ordering anyone ever sees is the top. So the task is to produce a priority score whose *top* is trustworthy; what happens at rank 4,000 is irrelevant.

Why not the others:

- **Not classification (as the headline task).** I could threshold the score into "needs review / doesn't," but that throws away the ordering the editor actually needs, and it forces me to pick a cutoff before I know their capacity. Capacity changes week to week; a ranking survives that, a fixed class boundary doesn't. Classification does appear *inside* my design as a means to an end — a model that outputs a probability gives me something to sort by — but the deliverable is the sorted list.
- **Not clustering.** Clustering answers "what kinds of pages exist," which is Lane 3's question. I already know the kind of page I care about (visible, under-capturing clicks); I need them *ordered*, and clustering gives no order.
- **Not regression, quite.** I do estimate a continuous quantity (how far below expected a page's CTR sits), but the number is only ever used to sort. I care about getting the ordering right at the top, not about minimising error at rank 4,000 — so I will judge it with a ranking metric, not R².

**The honest complication, stated up front.** "Ranking" describes the output. It does not by itself make this machine learning. As sections 2 and 5 show, the ranking I can build from the starter data today is **deterministic arithmetic** — an expected-CTR curve and a subtraction — with no learning in it at all. That version is a *baseline*, and it may be all this problem needs. The genuine ML question only appears when the target becomes a future observed outcome, and that needs the warehouse. I would rather name that now than dress up a subtraction as a model.

In [1]:
# Setup, and the shape of the decision that picks the task type.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

import numpy as np, pandas as pd
pd.set_option("display.width", 160)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same universe as ML-02, same reasons:
#   impressions_90d > 0 and content_age_days >= 90   (starter prep rules, lane guide S5)
#   avg_position > 0                                 (0 means "no position data", NOT rank zero)
#   impressions_90d >= 500, avg_position <= 20       (visible enough that a click is possible)
elig = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].drop_duplicates("content_id")
vis = elig[(elig.impressions_90d >= 500) & (elig.avg_position > 0) & (elig.avg_position <= 20)].copy()

CAPACITY = 50
print("The decision, in numbers:")
print(f"  candidate pages   : {len(vis):,}")
print(f"  reviewer capacity : {CAPACITY}")
print(f"  fraction ever seen: {CAPACITY/len(vis):.2%}")
print()
print("A yes/no label would answer a question nobody asks.")
print(f"An ORDER answers the real one: which {CAPACITY} of {len(vis):,} do I open first?")
print("-> task type = ranking / scoring. Metric must therefore live at the top of the list.")

The decision, in numbers:
  candidate pages   : 12,023
  reviewer capacity : 50
  fraction ever seen: 0.42%

A yes/no label would answer a question nobody asks.
An ORDER answers the real one: which 50 of 12,023 do I open first?
-> task type = ranking / scoring. Metric must therefore live at the top of the list.


## 2. Target or proxy

Two targets, and I need to keep them apart, because only one of them is real.

### The proxy I can build today (defined, not observed)

```text
expected_ctr = median CTR of pages in the same position band
ctr_gap      = ctr - expected_ctr          <- negative means under-capturing
```

**Where does this label come from? A rule I wrote.** That matters, and the framing skill is blunt about it: *"The target must be observed, not defined. A label that comes from someone's rule means your model learns the rule, not the world."* My `ctr_gap` is a **derived measurement** (lane guide §3) — built from an observed quantity (`ctr`), but normalized against a comparison group I chose. If I train a model to predict `ctr_gap`, the best it can do is rediscover my own expected-CTR curve. That is not discovery; it is an echo.

So `ctr_gap` is **not a learning target**. It is the arithmetic of the baseline itself. I am naming it here precisely because it looks like a target and isn't one — that confusion is the trap I would otherwise walk into in Week 4.

### The target I actually want (observed, needs the warehouse)

```text
features from a prior window   ->   observed outcome in a strictly later window
```

Concretely: *did this page's clicks rise over the next 30 days, holding position roughly constant?* That label is **observed** — it is a measurement of what happened, not a rule I applied. It is the only version that can honestly answer "was ranking this page worth an editor's hour?"

Two design problems I already know it has, and won't hand-wave:

1. **Position drift confounds it.** A page can gain clicks purely by ranking better, with no editorial merit at all. If I don't hold position roughly constant between windows, I will be measuring luck and calling it skill.
2. **The starter CSV cannot supply it.** As established in ML-02, the `last_30d`/`prev_30d` columns are cut from the *same* 90-day snapshot as `ctr` — a "prev vs last" comparison inside one snapshot describes the past; it is not a future outcome measured after a decision point. The code below re-checks this rather than taking my own earlier word for it.

### What that leaves me with, honestly

| | Proxy (`ctr_gap`) | Real target (future clicks) |
|---|---|---|
| Source | a rule I defined | an observed outcome |
| Available in starter CSV | yes | **no** |
| Can a model learn it? | only by echoing my curve | yes, genuinely |
| Honest claim | "under-captured clicks last quarter" | "was worth reviewing" |
| Status | descriptive baseline | the actual capstone target |

For ML-03 I frame both and build the proxy, because the proxy is a legitimate **baseline** and the card asks me to sketch the target column. But I will not call it a prediction, and I will not report a model score against it as evidence of anything — section 5 shows exactly why that would be self-deception.

In [2]:
# Build the proxy target column, and show why it is a proxy and not a target.

# Fine position bands beat the coarse position_tier: tiers lump position 1 with position 10.
vis["pos_band"] = vis.avg_position.round().clip(1, 20).astype(int)

# Expected CTR = what pages at THIS rank typically get. Median, not mean: CTR is skewed.
vis["expected_ctr"] = vis.groupby("pos_band")["ctr"].transform("median")
vis["ctr_gap"] = vis.ctr - vis.expected_ctr           # negative = under-capturing

print("=== the proxy target column, sketched ===")
print(vis["ctr_gap"].describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
print(f"\n  pages below their band's expected CTR: {(vis.ctr_gap < 0).sum():,} "
      f"({(vis.ctr_gap < 0).mean():.0%})")
print("  (~half by construction -- it is a median. That is a REMINDER that this")
print("   column is defined, not discovered.)")
print()

print("=== is ctr_gap observed, or defined? Trace its ancestry. ===")
print("  ctr_gap      <- ctr - expected_ctr")
print("  expected_ctr <- median(ctr) within a band I chose")
print("  ctr          <- clicks_90d / impressions_90d")
print("  -> every ingredient is the SAME 90d window. Nothing here is a future outcome.")
print("  -> VERDICT: derived measurement (lane guide S3), usable as a BASELINE,")
print("     not as a learning target.")
print()

# Re-check the ML-02 claim rather than trusting it: is there any forward window?
print("=== does the starter CSV contain a future outcome? ===")
win = [c for c in df.columns if "last_30" in c or "prev_30" in c]
print(f"  window-ish columns : {win}")
print("  All are slices of the SAME 90d snapshot that produced ctr.")
print("  'last_30 vs prev_30' compares two halves of the past -- there is no")
print("  window that begins AFTER a decision point.")
print("  -> VERDICT: no observed forward label available here. Warehouse required.")

=== the proxy target column, sketched ===
count    12023.000
mean         0.096
std          0.339
min         -0.330
10%         -0.180
25%         -0.110
50%          0.000
75%          0.190
90%          0.480
max          5.260

  pages below their band's expected CTR: 5,898 (49%)
  (~half by construction -- it is a median. That is a REMINDER that this
   column is defined, not discovered.)

=== is ctr_gap observed, or defined? Trace its ancestry. ===
  ctr_gap      <- ctr - expected_ctr
  expected_ctr <- median(ctr) within a band I chose
  ctr          <- clicks_90d / impressions_90d
  -> every ingredient is the SAME 90d window. Nothing here is a future outcome.
  -> VERDICT: derived measurement (lane guide S3), usable as a BASELINE,
     not as a learning target.

=== does the starter CSV contain a future outcome? ===
  window-ish columns : ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
  All are sl

## 3. Success metric

**The metric: precision@50.**

Of the 50 pages my ranking puts at the top, how many turn out to genuinely deserve the review? K = 50 because that is one reviewer's realistic weekly capacity, not because it makes the number look good. I am naming it now, before any training — the framing skill's rule, and a fair one: *"'Good' defined after the fact always looks good."*

**Why not the alternatives:**

- **Not accuracy.** ~12,000 pages, and a reviewer sees 50 of them. Accuracy would be dominated by the 11,950 rows nobody ever opens. I could score 90% accuracy and have a useless top-50.
- **Not ROC-AUC as the headline.** AUC asks whether the whole ordering is sensible. Nobody reads the whole ordering. AUC is worth reporting as a secondary sanity check, but optimizing it would trade top-of-list quality for middle-of-list tidiness — precisely the wrong trade.
- **Not R² on `ctr_gap`.** The number only exists to sort by. Being accurate at rank 4,000 has no cash value.

**What number would mean "good"?** I will not invent a target figure now, because the honest threshold is *relative*: the ranking has to beat the transparent baseline's precision@50 by enough to justify the extra complexity, and it has to beat it on a **forward-looking label**. For reference, the starter pipeline's own numbers on its (different, weaker) target were precision@50 of 0.240 for rules and 0.740 for a random forest — a useful sense of scale, but not my benchmark, since my target and universe both differ. If a transparent rule matches the model, the rule ships. "The simple thing won" is a result.

**The catch, and it is a big one.** Precision@K needs a label that says whether a page *deserved* the review. My proxy `ctr_gap` cannot say that — it says the page under-captured clicks last quarter, which is the same information the ranking was built from. Computing precision@50 against `ctr_gap` therefore grades my ranking against its own input. The code below computes the metric anyway, to prove it is mechanically available and to show exactly how flattering it looks — which is the point. **Precision@50 is my metric; it only becomes meaningful once the forward label from ML-02's next task exists.**

In [3]:
# Prove the metric is computable today (framing skill: "Can you compute it today, on a
# baseline? Do it."), and show how flattering it is against a proxy label.

def precision_at_k(scores, labels, k=50):
    """Of the top-k pages by score, what fraction carry label == 1?"""
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

# A proxy 'deserved review' label: worst quartile of ctr_gap WITHIN its position band.
# Per-band, so it can never just mean "ranks badly".
band_q25 = vis.groupby("pos_band")["ctr_gap"].transform(lambda s: s.quantile(0.25))
vis["underperf_proxy"] = (vis.ctr_gap <= band_q25).astype(int)
base_rate = vis.underperf_proxy.mean()

print(f"proxy label base rate: {base_rate:.3f}  (random ranking would score ~this at K=50)")
print()

# Baseline A: the starter's flat rule -- low CTR, ignore position entirely.
flat_rule = -vis.ctr
# Baseline B: my position-adjusted score -- rank by how far below the band a page sits.
adjusted = -vis.ctr_gap

print("precision@50 against the PROXY label:")
print(f"  random (base rate)                : {base_rate:.3f}")
print(f"  flat rule       (score = -ctr)    : {precision_at_k(flat_rule, vis.underperf_proxy):.3f}")
print(f"  position-adjust (score = -ctr_gap): {precision_at_k(adjusted, vis.underperf_proxy):.3f}")
print()
print("Both score ~1.0 -- and that is NOT good news. Read it carefully:")
print("  the label is derived from ctr_gap, and both scores are built from ctr.")
print("  I am grading the ranking against its own input. A perfect score here")
print("  measures arithmetic consistency, not usefulness to an editor.")
print()
print("-> The metric MECHANISM works (that is what this cell proves).")
print("   The metric MEANING waits on a forward-looking label from the warehouse.")

proxy label base rate: 0.262  (random ranking would score ~this at K=50)

precision@50 against the PROXY label:
  random (base rate)                : 0.262
  flat rule       (score = -ctr)    : 1.000
  position-adjust (score = -ctr_gap): 1.000

Both score ~1.0 -- and that is NOT good news. Read it carefully:
  the label is derived from ctr_gap, and both scores are built from ctr.
  I am grading the ranking against its own input. A perfect score here
  measures arithmetic consistency, not usefulness to an editor.

-> The metric MECHANISM works (that is what this cell proves).
   The metric MEANING waits on a forward-looking label from the warehouse.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (one page), observed over a trailing 90-day window, filtered to pages that are visible in search.**

In plain words: *one row is one page an editor could open and edit this week.* That is the grain, and it is chosen to match the action. Not one row per client (a client is not something you edit). Not one row per day (an editor doesn't act on a Tuesday). Not one row per query (the data ships no raw queries, and you can't edit a query).

The filters that define "in scope," and the reason for each:

| Filter | Why |
|---|---|
| `impressions_90d > 0`, `content_age_days >= 90` | the starter prep rules (lane guide §5) — a page needs history to judge |
| `impressions_90d >= 500` | below this, CTR is noise; a handful of clicks swings it wildly |
| `avg_position > 0` | **0 means "no position data", not rank zero** (data skill §1) — 1,205 such rows exist, and they cannot be compared to a position band |
| `avg_position <= 20` | beyond ~page 2 the realistic action is "improve rankings," not "rewrite the title" — a different lane's problem |
| `drop_duplicates("content_id")` | enforce the grain rather than assume it |

The code below shows the actual dataframe, then **probes the grain instead of trusting it** — if `content_id` weren't unique, every per-page number in this notebook would be silently wrong.

### What printing the dataframe actually taught me

I nearly didn't look. The rendered table is sorted by worst `ctr_gap`, and every page at the top had **zero clicks** despite thousands of impressions at position ~4. That prompted a check I hadn't planned, and it is the most consequential thing in this notebook:

- **10.1% of my visible universe (1,215 pages) received zero clicks in 90 days** — one of them with 208,678 impressions.
- **86% of a top-50 queue ranked by `ctr_gap` would be these zero-click pages**, drawn from just **9 of 28 clients**.
- The zero-click rate ranges from **43.5% for one client to near zero for others.**

A page ranking at position 4 with thousands of impressions and *literally zero* clicks over three months is not a plausible content problem — that is what broken click tracking looks like. And the fact that it clusters by client is the tell: editorial quality doesn't vary by client that sharply, but analytics configuration does.

This matters because it would have **silently wrecked the deliverable**. My queue's entire top would have been measurement artifacts from a handful of clients, and precision@50 would have looked fine while sending an editor to review pages that were never broken. It also reframes open question 2 from ML-02: the "client heterogeneity" I saw in the model's fold variance may be tracking heterogeneity, not content heterogeneity.

I am not fixing it here — ML-03 is framing, and the fix (a minimum-clicks floor, a per-client tracking-sanity check, or excluding zero-click pages entirely) is a **data-contract decision that belongs in ML-04**, where I can state and defend the threshold. But the grain discussion is exactly where it surfaced, so it is recorded here.

### One thing the grain hides

A single row averages a page's CTR across every query it ranks for, over 90 days. A page sitting at "average position 6" might rank 2nd for one query and 15th for another; its "gap" is then an artefact of that mix, not a fact about the page. The starter CSV cannot see this — it has no per-query breakdown. The warehouse's `fact_content_query_90d` table can (grain: client × content × query hash), so this is a real check I can run later, not a caveat I'm parking forever.

Missingness matters too, and it is **not** random: it tracks `content_type`. The code shows the pattern. A blind `fillna(0)` would quietly encode "which content type is this" into every feature — the trap the data skill names (§1). Has-flags, not fillna.

In [4]:
# The unit of analysis, made concrete.

print("=== GRAIN PROBE: is one row really one content item? ===")
dupes = vis.groupby("content_id").size()
print(f"  rows                      : {len(vis):,}")
print(f"  distinct content_id       : {vis.content_id.nunique():,}")
print(f"  content_ids with >1 row   : {(dupes > 1).sum()}   <- must be 0")
assert (dupes > 1).sum() == 0, "grain broken: content_id is not unique"
print("  -> grain confirmed: one row = one page.")
print()

print("=== how the universe was narrowed ===")
print(f"  all rows in CSV                    : {len(df):,}")
print(f"  eligible (imp>0, age>=90, dedup)   : {len(elig):,}")
print(f"  of those, no position data (==0)   : {(elig.avg_position == 0).sum():,}  (excluded)")
print(f"  MY UNIT UNIVERSE (visible pages)   : {len(vis):,}  across {vis.client_id.nunique()} clients")
print()

print("=== missingness is NOT random -- it tracks content_type ===")
miss = df.groupby("content_type")[["word_count", "search_volume", "cpc"]].apply(
    lambda g: (g.isna().mean() * 100).round(1))
print(miss.to_string())
print("  -> 'feedly article' is ~100% missing on keyword fields; 'keyword article'")
print("     is ~28% missing on word_count. A blind fillna(0) would smuggle")
print("     content_type into every feature. Use has-flags instead (data skill S1).")
print()

print("=== ONE ROW = ONE WHAT? Here it is. ===")
show = ["content_id", "client_id", "content_type", "avg_position", "pos_band",
        "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "ctr_gap"]
print("columns:", show)
print("(content_id / client_id are pseudonyms -- group and split keys only, never features)")
print()

# --- The check the dataframe above provoked. Every worst-gap row had clicks_90d == 0. ---
zero = vis[vis.clicks_90d == 0]
print("=== WAIT: the worst-gap rows all have ZERO clicks. How common is that? ===")
print(f"  visible pages with 0 clicks in 90d : {len(zero):,} / {len(vis):,} "
      f"({len(zero)/len(vis):.1%})")
print(f"  their median impressions_90d       : {zero.impressions_90d.median():,.0f}")
print(f"  MOST impressions on a 0-click page : {zero.impressions_90d.max():,.0f}   <- implausible")
print()

zero_share = (vis.assign(z=(vis.clicks_90d == 0))
                 .groupby("client_id")["z"].mean().sort_values(ascending=False))
print("  zero-click share by client (top 5, %):")
print((zero_share.head(5) * 100).round(1).to_string())
print(f"  ...ranging down to {zero_share.min()*100:.1f}% for the cleanest client.")
print()

top50 = vis.nsmallest(50, "ctr_gap")
print("  If I shipped a top-50 queue by ctr_gap TODAY:")
print(f"    share that are zero-click pages  : {(top50.clicks_90d == 0).mean():.0%}")
print(f"    distinct clients represented     : {top50.client_id.nunique()} of {vis.client_id.nunique()}")
print()
print("  -> A page at position ~4 with thousands of impressions and zero clicks over")
print("     90 days is not a content problem; that is what broken click tracking")
print("     looks like. It clusters BY CLIENT, which is the tell -- editorial quality")
print("     does not vary that sharply between clients, but analytics setup does.")
print("  -> Unhandled, this would make my queue's top 50 mostly measurement artifacts.")
print("     The fix (min-clicks floor / per-client tracking check) is a DATA CONTRACT")
print("     decision -> ML-04, where I can state and defend the threshold.")
print()

vis[show].sort_values("ctr_gap").head(8)

=== GRAIN PROBE: is one row really one content item? ===
  rows                      : 12,023
  distinct content_id       : 12,023
  content_ids with >1 row   : 0   <- must be 0
  -> grain confirmed: one row = one page.

=== how the universe was narrowed ===
  all rows in CSV                    : 30,000
  eligible (imp>0, age>=90, dedup)   : 30,000
  of those, no position data (==0)   : 1,205  (excluded)
  MY UNIT UNIVERSE (visible pages)   : 12,023  across 28 clients

=== missingness is NOT random -- it tracks content_type ===
                    word_count  search_volume    cpc
content_type                                        
comparison article         0.0            0.0    0.0
feedly article             0.0          100.0  100.0
keyword article           28.3            1.4    1.4
  -> 'feedly article' is ~100% missing on keyword fields; 'keyword article'
     is ~28% missing on word_count. A blind fillna(0) would smuggle
     content_type into every feature. Use has-flags inste

,content_id,client_id,content_type,avg_position,pos_band,impressions_90d,clicks_90d,ctr,expected_ctr,ctr_gap
5732,content_b893db8d7619,client_3fdba35f04,keyword article,4.5,4,818,0,0.0,0.33,-0.33
7159,content_4b1303a5affe,client_7f2253d7e2,keyword article,4.1,4,1341,0,0.0,0.33,-0.33
7586,content_ce8619672faf,client_19581e27de,keyword article,3.5,4,3855,0,0.0,0.33,-0.33
9092,content_69deabcc81b5,client_19581e27de,keyword article,4.5,4,1961,0,0.0,0.33,-0.33
4533,content_d6e1bbb4a996,client_d029fa3a95,keyword article,3.9,4,4955,0,0.0,0.33,-0.33
10326,content_f658e746131f,client_4e07408562,keyword article,3.5,4,1485,0,0.0,0.33,-0.33
4068,content_0ad759bc5d3d,client_19581e27de,keyword article,4.3,4,3409,0,0.0,0.33,-0.33
10455,content_833697cdd9e3,client_6208ef0f77,keyword article,3.6,4,752,0,0.0,0.33,-0.33


## 5. Why ML beats a fixed rule here

I tested this instead of asserting it, and the answer is **"partly, and not yet provably"** — which is more useful than a confident yes.

### What a fixed rule gets wrong, provably

The starter's rule is `ctr < 0.5 AND impressions >= 500` — a flat cutoff. It is wrong in a way I can demonstrate rather than argue: expected CTR **depends on rank**. Median CTR runs ≈0.33% at position 4 and ≈0.17% at position 10 (measured in ML-02). A flat threshold therefore condemns almost every deep-ranking page and forgives almost every shallow one. It isn't finding under-performers; it's re-discovering rank. That much is settled, and it justifies the position adjustment.

But here is the thing: **the position adjustment is itself a rule.** A median per band and a subtraction. No learning. If the story ended there, the honest answer to this section would be *"ML doesn't beat a rule — a better rule beats a worse rule,"* and I would ship the arithmetic.

### Where learning does add something

So I tested whether observable features carry information about which pages under-capture clicks, using a random forest that **never sees `ctr` or `clicks_90d`** (only position, impressions, word count, age, freshness, engagement, and has-flags), validated with a **client-grouped** split so it can't memorize a client:

- **ROC-AUC ≈ 0.77** on the per-band underperformer proxy, against 0.5 for chance.

That is real, and it is the argument for ML: something about a page's observable characteristics predicts under-capture, and it isn't a single threshold — no one column carries it. A hand-written if-statement over 13 signals with interactions is just a decision tree that nobody validated.

### Three reasons I am not declaring victory

1. **The fold spread is large.** AUC ranges 0.70–0.83 across client folds, and the continuous version (R² on `ctr_gap`) swings from **−0.08 to +0.36** — *negative* on one client group, meaning worse than predicting the mean. The pattern is client-dependent, and 28 clients is not many. Whatever this model learns, it does not transfer evenly.
2. **The categorical signals are nearly flat.** Median `ctr_gap` shifts only 0.02–0.05pp across intent, word-count tier, and freshness tier — noise-level, against a gap whose own standard deviation is 0.34. And 98.5% of rows are one content type, so `content_type` is close to a constant here. The multi-signal story is thinner than I expected.
3. **The comparison is rigged, and not in the model's favour.** This is the one that matters. The flat rule scores **AUC 0.983** on this proxy label — not because it is excellent, but because the label is derived from `ctr` and the rule *reads* `ctr`. It is peeking at the answer. My model, denied that column, gets 0.77 and looks worse while actually being the only one doing work.

### The conclusion I will defend

**On the starter data, "rule vs ML" is not decidable — the question is malformed, because any label I can build here is made of the same column the rule reads.** That is not a gap in my effort; it is a structural property of a single-snapshot dataset. The comparison only becomes meaningful against a **forward-looking observed label**, where the rule has no privileged access to the answer and both methods must actually predict.

So my defensible claim for now: *a position-adjusted ranking is demonstrably better-founded than a flat CTR cutoff, and there is measurable signal (AUC ≈0.77, client-grouped) suggesting a model can add more — but whether it earns its complexity is an open question that the warehouse, not this file, will settle.* If the rule wins there, I ship the rule.

In [5]:
# Test the "ML beats a rule" claim instead of asserting it.
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, r2_score

# SAFE features only. Deliberately EXCLUDED: ctr, clicks_90d (the label's parents),
# trend_direction / trend_pct (the starter label's ancestors -- data skill S1).
FEATS = ["avg_position", "impressions_90d", "search_volume", "competition", "cpc",
         "word_count", "char_count", "content_age_days", "days_since_last_update",
         "sessions_90d", "engagement_rate", "scroll_rate", "days_with_impressions"]

X = vis[FEATS].copy()
for c in FEATS:                                   # has-flags, NOT a blind fillna
    X[c + "_isna"] = X[c].isna().astype(int)
X = X.fillna(X.median(numeric_only=True))
y_bin = vis.underperf_proxy.values
y_num = vis.ctr_gap.values
groups = vis.client_id.values                     # client-grouped: no client in both sides

gkf = GroupKFold(n_splits=5)
aucs, r2s = [], []
for tr, te in gkf.split(X, y_bin, groups):
    clf = RandomForestClassifier(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=-1)
    clf.fit(X.iloc[tr], y_bin[tr])
    aucs.append(roc_auc_score(y_bin[te], clf.predict_proba(X.iloc[te])[:, 1]))
    reg = RandomForestRegressor(n_estimators=120, min_samples_leaf=20, random_state=0, n_jobs=-1)
    reg.fit(X.iloc[tr], y_num[tr])
    r2s.append(r2_score(y_num[te], reg.predict(X.iloc[te])))

print("=== Can safe features (NO ctr, NO clicks) predict under-capture? ===")
print(f"  ROC-AUC per client fold : {[round(a, 3) for a in aucs]}")
print(f"  ROC-AUC mean            : {np.mean(aucs):.3f}   (chance = 0.500)")
print("  -> Yes: there IS real signal in observable features. Argument FOR ML.")
print()
print(f"  R^2 on ctr_gap per fold : {[round(r, 3) for r in r2s]}")
print(f"  R^2 mean                : {np.mean(r2s):.3f}")
print("  -> But it swings NEGATIVE on one client group (worse than the mean).")
print("     The pattern is client-dependent, and 28 clients is not many.")
print()

print("=== The trap: what does the FLAT RULE score on this same label? ===")
auc_flat = roc_auc_score(y_bin, -vis.ctr)
print(f"  flat rule  (score = -ctr)      : AUC {auc_flat:.3f}")
print(f"  my model   (never sees ctr)    : AUC {np.mean(aucs):.3f}")
print()
print("  The rule 'wins' -- because the label was DERIVED from ctr and the rule")
print("  READS ctr. It is peeking at the answer key, not predicting.")
print("  -> On this data, 'rule vs ML' cannot be settled. The question is malformed.")
print("     Only a forward-looking observed label can settle it, where neither")
print("     method can see the outcome in advance. That is the warehouse task.")
print()

imp = (pd.Series(clf.feature_importances_, index=X.columns)
         .sort_values(ascending=False).head(6))
print("=== which safe signals carry the AUC 0.77? (last fold, directional only) ===")
print(imp.round(3).to_string())

=== Can safe features (NO ctr, NO clicks) predict under-capture? ===
  ROC-AUC per client fold : [0.698, 0.785, 0.832, 0.722, 0.832]
  ROC-AUC mean            : 0.774   (chance = 0.500)
  -> Yes: there IS real signal in observable features. Argument FOR ML.

  R^2 on ctr_gap per fold : [-0.075, 0.209, 0.359, 0.064, 0.293]
  R^2 mean                : 0.170
  -> But it swings NEGATIVE on one client group (worse than the mean).
     The pattern is client-dependent, and 28 clients is not many.

=== The trap: what does the FLAT RULE score on this same label? ===
  flat rule  (score = -ctr)      : AUC 0.983
  my model   (never sees ctr)    : AUC 0.774

  The rule 'wins' -- because the label was DERIVED from ctr and the rule
  READS ctr. It is peeking at the answer key, not predicting.
  -> On this data, 'rule vs ML' cannot be settled. The question is malformed.
     Only a forward-looking observed label can settle it, where neither
     method can see the outcome in advance. That is the ware

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — the starter CSV has no URL/domain/title/query columns; only pseudonymous IDs and aggregates appear, and the sample dataframe shows scrambled `content_id`/`client_id` only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### The ML loop, filled in

| Piece | My answer |
|---|---|
| **Task type** | Ranking / scoring (a probability is a means; the sorted list is the deliverable) |
| **Unit of analysis** | One visible page — `content_id`, 90-day window, 12,023 rows, 28 clients |
| **Proxy (today)** | `ctr_gap` = ctr − median CTR of its position band — **defined, not observed**; a baseline, not a target |
| **Real target (capstone)** | Clicks rose over the next 30 days, position held roughly constant — **observed**; needs the warehouse |
| **Success metric** | precision@50 (K = one reviewer's week); AUC secondary; never accuracy |
| **Action it supports** | An editor rewrites the title/meta, fixes intent match, improves snippet structure — or monitors |
| **Why not just a rule** | A flat CTR cutoff provably re-discovers rank. A position adjustment fixes that but is itself a rule. Learning adds measurable signal (AUC ≈0.77, client-grouped, no `ctr`), but its value is **unproven** until a forward label exists |

### What I actually learned writing this

Two things, and neither was what I set out to find.

**The comparison I planned is malformed.** I meant to justify ML over a rule. Instead: any label I can build from this file is made from `ctr`, and the rule I'd compare against *reads* `ctr` — so the rule scores 0.983 AUC by peeking while my model scores 0.774 by actually working. That is not a close call to break with a better model; it is a structural property of a single-snapshot dataset. Naming it is worth more than a leaderboard number I couldn't defend.

**Printing the dataframe caught a bug my metrics never would have.** Every worst-gap page had zero clicks. It turns out 10.1% of my universe has zero clicks in 90 days, one with 208,678 impressions, clustered in a handful of clients (43.5% for the worst, near zero for the best) — and **86% of a top-50 queue would be these pages, from 9 of 28 clients.** Every aggregate in this notebook looked healthy while the actual deliverable was quietly full of tracking artifacts. Precision@50 would have *rewarded* me for it. That is the lesson: the summary statistics were all fine, and the eight rows I printed were not.

### Open questions carried into ML-04

1. **Zero-click pages — the one that must be settled first.** Are these broken tracking or real content failures? The 208,678-impression case says tracking; the per-client clustering says tracking. The contract needs a defensible rule (minimum clicks? per-client tracking sanity check? exclude entirely?) and I need to state what each choice costs. Until this is resolved, my ranking is not shippable.
2. **Client heterogeneity may be tracking heterogeneity.** R² went *negative* on one client fold. I assumed content differences; the zero-click finding suggests the model may be partly learning which clients measure clicks properly. That would be a real leak — measurement quality masquerading as signal.
3. **Is the position band the right comparison group?** A page's "average position 6" may average rank 2 on one query and rank 15 on another, making its gap an artefact of query mix. `fact_content_query_90d` (client × content × query hash) can test this; the starter CSV structurally cannot.
4. **How do I hold position constant in the forward label?** Clicks rising because a page ranked better is not editorial merit. The hardest design problem in my lane, and it belongs in the data contract, not in a model.
5. **The `top_3` inversion from ML-02 is still unexplained** (median CTR 0.20% vs page_1's 0.24%, n=458). Given finding 1, I now suspect zero-click contamination rather than a real fact about top-3 queries — worth re-checking once the floor is set.